# Experiment 43 — Backward-free SparseWalker on ML-1M (in-process fast path)

Runs inside the already-live Colab kernel: no second Python/PyTorch process. Learning rules and architecture are unchanged.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, json, torch, importlib
from pathlib import Path
REPO='/content/Sparsewalker'
BRANCH='research/active'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO],check=True)
for p in [f'{REPO}/src', f'{REPO}/experiments']:
    if p not in sys.path: sys.path.insert(0,p)
importlib.invalidate_caches()
import sparsewalker
HEAD=subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip()
print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,flush=True)
print('TORCH',torch.__version__,flush=True)
print('BRANCH',BRANCH,flush=True)
print('HEAD',HEAD,flush=True)
print('SPARSEWALKER_IMPORT',sparsewalker.__file__,flush=True)
assert torch.cuda.is_available(), 'GPU runtime required'


## Run

This uses `runpy` inside the current kernel, avoiding the child-interpreter stall. You should see `SCRIPT_OK`, then `ML1M_DATA_LOAD_START`, `ML1M_DATA_LOAD_DONE`, `ML1M_TRAIN_START`, and batch progress.


In [ ]:
import runpy, sys, os, importlib
RESUME=False
EPOCHS=70
SCRIPT=f'{REPO}/experiments/run_ml1m_local_contrastive_walker.py'
for p in [f'{REPO}/src', f'{REPO}/experiments']:
    if p not in sys.path: sys.path.insert(0,p)
importlib.invalidate_caches()
import sparsewalker
print('IMPORT_OK', sparsewalker.__file__, flush=True)
text=Path(SCRIPT).read_text()
assert 'Experiment 43 v3' in text and 'ML1M_DATA_LOAD_START' in text, 'stale script clone'
print('SCRIPT_OK',SCRIPT,flush=True)
argv=[SCRIPT,
      '--epochs',str(EPOCHS),
      '--batch-size','512',
      '--eval-batch-size','1024',
      '--eval-every','1',
      '--progress-every','1',
      '--data-dir','/content/drive/MyDrive/sparsewalker_data']
if RESUME: argv.append('--resume')
print('RUNNING_IN_PROCESS',' '.join(argv),flush=True)
old_argv=sys.argv[:]
old_cwd=os.getcwd()
sys.argv=argv
os.chdir(REPO)
try:
    runpy.run_path(SCRIPT,run_name='__main__')
finally:
    sys.argv=old_argv
    os.chdir(old_cwd)


## Inspect trajectory


In [ ]:
import pandas as pd
root=Path('/content/drive/MyDrive/sparsewalker_local_contrastive_ml1m/seed42')
hp=root/'history.json'
if hp.exists():
    h=pd.DataFrame(json.loads(hp.read_text()))
    cols=['epoch','mean_contrastive_margin','mean_positive_prob','mean_negative_prob','mean_router_confidence','mean_value_update','mean_context_error','val_NDCG@10','val_HR@10','val_MRR@10','positions_per_s','padding_efficiency']
    display(h[[c for c in cols if c in h.columns]])
    if len(h):
        best=h.loc[h['val_NDCG@10'].idxmax()]
        print('BEST',best.to_dict())
else:
    print('No history yet.')


## Final / crash recovery status


In [ ]:
rp=root/'result.json'
bp=root/'best.pt'
lp=root/'last.pt'
print('best.pt',bp.exists(),'last.pt',lp.exists(),'result.json',rp.exists())
if rp.exists():
    print(json.dumps(json.loads(rp.read_text()),indent=2))
elif lp.exists():
    ck=torch.load(lp,map_location='cpu')
    print('RECOVERABLE_FROM_EPOCH',ck['epoch'],'BEST_EPOCH',ck.get('best_epoch'),'BEST_VAL',ck.get('best'))
